# BirdML — Final Training (notebook version of `run_train.py`)

Self-contained version of the SageMaker training step, built to run inside a SageMaker Studio notebook with a GPU instance. Replicates `infrastructure/sagemaker/scripts/run_train.py` end-to-end:

1. Sync raw dataset from S3 (or HF if missing) and load splits
2. Build transforms, label mapping, BirdDataset, and DataLoaders
3. Pull best hyperparameters from S3 (fall back to defaults)
4. Build EfficientNet-B3 with ImageNet pretrained weights pulled from S3
5. Run two-stage training (warmup + finetune), log to MLflow
6. Upload checkpoint, class_names.json, and mlruns/ to S3

## 1. Setup: imports, paths, and constants

In [5]:
import json
import logging
from pathlib import Path
from typing import Any, Callable

import boto3
import mlflow
import torch
import torch.nn as nn
import torch.optim as optim
from botocore.exceptions import ClientError
from datasets import DatasetDict, load_dataset
from huggingface_hub import snapshot_download
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import efficientnet_b3
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

# ─── AWS / S3 ────────────────────────────────────────────────────────────────
AWS_REGION = "us-east-2"
S3_BUCKET = "bird-ml-halajeel"
S3_DATA_PREFIX = "data/raw/birds-525"
S3_PARAMS_PREFIX = "params"
S3_MODELS_PREFIX = "models"
S3_MLRUNS_PREFIX = "mlruns"
S3_BEST_PARAMS_KEY = f"{S3_PARAMS_PREFIX}/best_params.json"

# ─── Local paths ─────────────────────────────────────────────────────────────
# Adjust if running outside /home/sagemaker-user.
WORK_DIR = Path.cwd()
DATA_DIR = WORK_DIR / "data" / "raw" / "birds-525"
MODELS_DIR = WORK_DIR / "models"
MLRUNS_DIR = WORK_DIR / "mlruns"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# ─── Dataset constants ───────────────────────────────────────────────────────
NUM_CLASSES = 526
IMAGE_SIZE = 224
RESIZE_BEFORE_CROP = 256

# ─── ImageNet normalization (required for pretrained EfficientNet) ───────────
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# ─── Augmentation ────────────────────────────────────────────────────────────
CROP_SCALE = (0.8, 1.0)
COLOR_JITTER_BRIGHTNESS = 0.2
COLOR_JITTER_CONTRAST = 0.2

# ─── Fixed optimizer / scheduler internals ───────────────────────────────────
RMS_MOMENTUM = 0.9
RMS_ALPHA = 0.9
RMS_EPS = 1e-8
SGD_MOMENTUM = 0.9
STEPLR_GAMMA = 0.1

# ─── Pretrained weights ──────────────────────────────────────────────────────
EFFICIENTNET_B3_WEIGHTS_FILENAME = "efficientnet_b3_rwightman-b3899882.pth"

# ─── Default + best hyperparameters (fallbacks if S3 is empty) ───────────────
DEFAULT_HYPERPARAMS = {
    "lr_head": 1e-3,
    "lr_backbone": 1e-4,
    "n_warmup_epochs": 3,
    "n_finetune_epochs": 12,
    "batch_size": 128,
    "weight_decay": 1e-5,
    "optimizer": "RMSprop",
    "scheduler": "StepLR",
}

BEST_PARAMS = {
    "lr_head": 0.0007500045272821452,
    "lr_backbone": 3.987986937649681e-05,
    "n_warmup_epochs": 4,
    "n_finetune_epochs": 11,
    "batch_size": 128,
    "weight_decay": 1e-5,
    "optimizer": "RMSprop",
    "scheduler": "StepLR",
}

HF_REPO_ID = "yashikota/birds-525-species-image-classification"
HF_REPO_TYPE = "dataset"

## 2. S3 + data ingestion helpers

In [3]:
def s3_prefix_exists(
    bucket: str = S3_BUCKET,
    prefix: str = S3_DATA_PREFIX,
    region: str = AWS_REGION,
) -> bool:
    """Return True if at least one object exists under the prefix."""
    s3 = boto3.client("s3", region_name=region)
    response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix, MaxKeys=1)
    return "Contents" in response


def download_from_hugging_face(local_dir: Path = DATA_DIR) -> Path:
    """Download the HuggingFace dataset snapshot to a local directory."""
    local_dir.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id=HF_REPO_ID,
        repo_type=HF_REPO_TYPE,
        local_dir=str(local_dir),
    )
    return local_dir


def upload_directory_to_s3(
    local_dir: Path,
    bucket: str = S3_BUCKET,
    prefix: str = S3_DATA_PREFIX,
    region: str = AWS_REGION,
) -> None:
    """Recursively upload every file under local_dir to s3://bucket/prefix/."""
    s3 = boto3.client("s3", region_name=region)
    for path in local_dir.rglob("*"):
        if path.is_file():
            relative_path = path.relative_to(local_dir)
            s3_key = f"{prefix}/{relative_path.as_posix()}"
            s3.upload_file(Filename=str(path), Bucket=bucket, Key=s3_key)


def download_directory_from_s3(
    local_dir: Path = DATA_DIR,
    bucket: str = S3_BUCKET,
    prefix: str = S3_DATA_PREFIX,
    region: str = AWS_REGION,
) -> Path:
    """Download an S3 prefix recursively into local_dir, preserving structure."""
    s3 = boto3.client("s3", region_name=region)
    local_dir.mkdir(parents=True, exist_ok=True)
    paginator = s3.get_paginator("list_objects_v2")
    found_any_object = False
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            found_any_object = True
            s3_key = obj["Key"]
            if s3_key.endswith("/"):
                continue
            relative_path = Path(s3_key).relative_to(prefix)
            local_path = local_dir / relative_path
            local_path.parent.mkdir(parents=True, exist_ok=True)
            s3.download_file(Bucket=bucket, Key=s3_key, Filename=str(local_path))
    if not found_any_object:
        raise FileNotFoundError(
            f"No objects found in s3://{bucket}/{prefix}. "
            "Dataset may not have been uploaded correctly."
        )
    return local_dir


def upload_file_to_s3(
    local_path: Path,
    bucket: str = S3_BUCKET,
    prefix: str = S3_MODELS_PREFIX,
    region: str = AWS_REGION,
) -> str:
    """Upload a single file to S3. Returns the S3 URI."""
    s3_key = f"{prefix}/{local_path.name}"
    boto3.client("s3", region_name=region).upload_file(
        str(local_path), bucket, s3_key
    )
    return f"s3://{bucket}/{s3_key}"


def ensure_dataset_available(
    local_dir: Path = DATA_DIR,
    bucket: str = S3_BUCKET,
    prefix: str = S3_DATA_PREFIX,
    region: str = AWS_REGION,
) -> Path:
    """Ensure the raw dataset exists in S3 and locally.

    If S3 has it, download locally (if not already). Else fetch from HF and upload to S3.
    """
    parquet_dir = local_dir / "data"
    if s3_prefix_exists(bucket=bucket, prefix=prefix, region=region):
        if not parquet_dir.exists():
            download_directory_from_s3(local_dir=local_dir, bucket=bucket, prefix=prefix, region=region)
        return local_dir
    download_from_hugging_face(local_dir=local_dir)
    upload_directory_to_s3(local_dir=local_dir, bucket=bucket, prefix=prefix, region=region)
    return local_dir


def load_bird_dataset(local_dir: Path = DATA_DIR, test_bool: bool = False) -> DatasetDict:
    """Load the bird dataset from local parquet files (syncing from S3/HF first if needed)."""
    ensure_dataset_available(local_dir=local_dir)
    parquet_data_dir = local_dir / "data"
    if not parquet_data_dir.exists():
        raise FileNotFoundError(
            f"Expected parquet data directory at {parquet_data_dir}, but it does not exist."
        )
    dataset = load_dataset("parquet", data_dir=str(parquet_data_dir))
    if test_bool:
        dataset = dataset.filter(lambda row: int(row["label"]) in {0, 1})
    return dataset

## 3. Load the dataset

In [8]:
# Set test_bool=True for a tiny 2-class smoke test
dataset = load_bird_dataset(test_bool=False)
logger.info("Loaded dataset: %s", dataset)

2026-05-13 18:53:21,670 INFO - HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/parquet/parquet.py "HTTP/1.1 404 Not Found"


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

2026-05-13 18:53:28,629 INFO - Loaded dataset: DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 84635
    })
    validation: Dataset({
        features: ['image', 'label'],
        num_rows: 2625
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 2625
    })
})


## 4. Image transforms, label mapping, and BirdDataset

In [6]:
def get_train_transforms() -> transforms.Compose:
    """Augmentations for training."""
    return transforms.Compose([
        transforms.RandomResizedCrop(IMAGE_SIZE, scale=CROP_SCALE),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=COLOR_JITTER_BRIGHTNESS, contrast=COLOR_JITTER_CONTRAST),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


def get_eval_transforms() -> transforms.Compose:
    """Deterministic transforms for val/test/inference."""
    return transforms.Compose([
        transforms.Resize(RESIZE_BEFORE_CROP),
        transforms.CenterCrop(IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


def build_label_mapping(dataset) -> dict[int, int]:
    """Map sparse original labels to dense 0..N-1 indices across all splits."""
    unique_labels = set()
    for split in ("train", "validation", "test"):
        unique_labels.update(int(label) for label in dataset[split]["label"])
    return {original_label: dense_idx for dense_idx, original_label in enumerate(sorted(unique_labels))}


def build_idx_to_name(dataset) -> dict[int, str]:
    """Build mapping from DENSE label index to species name.

    Mirrors the compaction done by build_label_mapping so class_names.json[i]
    is the species the trained model actually predicts at dense output i.
    """
    class_label = dataset["train"].features["label"]
    label_to_idx = build_label_mapping(dataset)
    return {
        dense_idx: class_label.int2str(original_label)
        for original_label, dense_idx in label_to_idx.items()
    }


def save_idx_to_name(idx_to_name: dict[int, str], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(idx_to_name, indent=2))


class BirdDataset(Dataset):
    """PyTorch wrapper around a HuggingFace dataset split: PIL -> tensor, sparse -> dense label."""

    def __init__(
        self,
        hf_split,
        transform: Callable | None = None,
        label_to_idx: dict[int, int] | None = None,
    ) -> None:
        self.hf_split = hf_split
        self.transform = transform
        self.label_to_idx = label_to_idx

    def __len__(self) -> int:
        return len(self.hf_split)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, int]:
        sample = self.hf_split[idx]
        image = sample["image"].convert("RGB")
        label = int(sample["label"])
        if self.label_to_idx is not None:
            label = self.label_to_idx[label]
        if self.transform is not None:
            image = self.transform(image)
        return image, label

## 5. Model: pretrained weights pulled from S3, EfficientNet-B3 builder

In [ ]:
def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def ensure_pretrained_weights_local(
    weights_path: Path,
    bucket: str = S3_BUCKET,
    key: str = f"{S3_MODELS_PREFIX}/{EFFICIENTNET_B3_WEIGHTS_FILENAME}",
    region: str = AWS_REGION,
) -> None:
    """Download EfficientNet-B3 ImageNet weights from S3 if not already on disk."""
    if weights_path.exists():
        return
    weights_path.parent.mkdir(parents=True, exist_ok=True)
    boto3.client("s3", region_name=region).download_file(
        Bucket=bucket, Key=key, Filename=str(weights_path)
    )


def build_model(
    num_classes: int = NUM_CLASSES,
    weights_path: Path | None = None,
) -> nn.Module:
    """Build EfficientNet-B3 with ImageNet backbone + fresh classification head."""
    if weights_path is None:
        weights_path = MODELS_DIR / EFFICIENTNET_B3_WEIGHTS_FILENAME
    ensure_pretrained_weights_local(weights_path)
    model = efficientnet_b3(weights=None)
    model.load_state_dict(
        torch.load(weights_path, map_location="cpu", weights_only=True)
    )
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

## 6. Training engine (run_epoch + top-k accuracy)

In [ ]:
def top_k_accuracy(outputs: torch.Tensor, labels: torch.Tensor, k: int) -> float:
    """Fraction of samples where the correct label is in the top-k predictions."""
    _, top_k_preds = outputs.topk(k, dim=1)
    correct = top_k_preds.eq(labels.view(-1, 1).expand_as(top_k_preds))
    return correct.any(dim=1).float().mean().item()


def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    optimizer: torch.optim.Optimizer | None = None,
    training: bool = True,
    epoch_label: str = "",
) -> tuple[float, float, float]:
    """Run one full pass over loader. Returns avg_loss, top1, top5 averaged over batches."""
    model.train(training)
    criterion = nn.CrossEntropyLoss()
    total_loss = total_top1 = total_top5 = 0.0
    pbar = tqdm(loader, desc=epoch_label, leave=False)
    with torch.set_grad_enabled(training):
        for images, labels in pbar:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item()
            total_top1 += top_k_accuracy(outputs, labels, k=1)
            total_top5 += top_k_accuracy(outputs, labels, k=5)
            pbar.set_postfix(loss=f"{loss.item():.4f}")
    n = len(loader)
    return total_loss / n, total_top1 / n, total_top5 / n

## 7. Optimizer + scheduler factories

In [ ]:
def build_optimizer(
    model: nn.Module,
    hyperparams: dict[str, Any],
    stage: str,
) -> optim.Optimizer:
    """Warmup: single group on unfrozen params at lr_head.
    Finetune: classifier at lr_head, backbone at lr_backbone.
    """
    name = hyperparams["optimizer"]
    lr_head = hyperparams["lr_head"]
    lr_backbone = hyperparams["lr_backbone"]
    weight_decay = hyperparams["weight_decay"]

    if stage == "warmup":
        param_groups = [p for p in model.parameters() if p.requires_grad]
    else:
        param_groups = [
            {"params": list(model.classifier.parameters()), "lr": lr_head},
            {"params": list(model.features.parameters()), "lr": lr_backbone},
        ]

    if name == "RMSprop":
        return optim.RMSprop(
            param_groups,
            lr=lr_head,
            momentum=RMS_MOMENTUM,
            alpha=RMS_ALPHA,
            eps=RMS_EPS,
            weight_decay=weight_decay,
        )
    elif name == "Adam":
        return optim.Adam(param_groups, lr=lr_head, weight_decay=weight_decay)
    elif name == "AdamW":
        return optim.AdamW(param_groups, lr=lr_head, weight_decay=weight_decay)
    elif name == "SGD":
        return optim.SGD(param_groups, lr=lr_head, momentum=SGD_MOMENTUM, weight_decay=weight_decay)
    else:
        raise ValueError(f"Unknown optimizer: {name!r}")


def build_scheduler(
    optimizer: optim.Optimizer,
    hyperparams: dict[str, Any],
    n_finetune: int,
) -> optim.lr_scheduler.LRScheduler:
    name = hyperparams["scheduler"]
    if name == "StepLR":
        return optim.lr_scheduler.StepLR(
            optimizer, step_size=max(1, n_finetune // 3), gamma=STEPLR_GAMMA
        )
    elif name == "CosineAnnealingLR":
        return optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_finetune)
    else:
        raise ValueError(f"Unknown scheduler: {name!r}")

## 8. Hyperparameters — using BEST_PARAMS from config

In [ ]:
# Use the hardcoded BEST_PARAMS from config — skips S3 entirely so stale
# smoke-test params in best_params.json don't interfere.
params = BEST_PARAMS
logger.info("Using BEST_PARAMS:\n%s", json.dumps(params, indent=2))

## 9. Build DataLoaders (using `batch_size` from params)

In [ ]:
def build_dataloaders(
    dataset: DatasetDict,
    label_to_idx: dict[int, int] | None = None,
    batch_size: int | None = None,
    num_workers: int = 0,
) -> tuple[DataLoader, DataLoader, DataLoader]:
    """Build train, validation, and test DataLoaders."""
    if label_to_idx is None:
        label_to_idx = build_label_mapping(dataset)
    if batch_size is None:
        batch_size = DEFAULT_HYPERPARAMS["batch_size"]

    train_dataset = BirdDataset(
        hf_split=dataset["train"], transform=get_train_transforms(), label_to_idx=label_to_idx,
    )
    val_dataset = BirdDataset(
        hf_split=dataset["validation"], transform=get_eval_transforms(), label_to_idx=label_to_idx,
    )
    test_dataset = BirdDataset(
        hf_split=dataset["test"], transform=get_eval_transforms(), label_to_idx=label_to_idx,
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    return train_loader, val_loader, test_loader


label_to_idx = build_label_mapping(dataset)
train_loader, val_loader, test_loader = build_dataloaders(
    dataset=dataset,
    label_to_idx=label_to_idx,
    batch_size=params["batch_size"],
)
logger.info(
    "DataLoaders ready — train batches=%d, val batches=%d, test batches=%d",
    len(train_loader), len(val_loader), len(test_loader),
)

## 10. Run two-stage training (warmup + finetune) with MLflow logging

In [ ]:
def run_training(
    dataset: DatasetDict,
    params: dict[str, Any] | None = None,
    mlflow_experiment: str = "bird_training",
    checkpoint_name: str = "EN_final.pth",
) -> Path:
    """Full two-stage training run with MLflow logging. Returns the checkpoint path."""
    if params is None:
        params = BEST_PARAMS

    device = get_device()
    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    checkpoint_path = MODELS_DIR / checkpoint_name

    label_to_idx = build_label_mapping(dataset)
    train_loader, val_loader, _ = build_dataloaders(
        dataset=dataset, label_to_idx=label_to_idx, batch_size=params["batch_size"],
    )

    n_warmup = params["n_warmup_epochs"]
    n_finetune = params["n_finetune_epochs"]

    mlflow.set_experiment(mlflow_experiment)
    with mlflow.start_run(run_name="final_training"):
        mlflow.log_params(params)
        model = build_model().to(device)

        # ─── Stage 1: Warmup ─────────────────────────────────────────────────
        for p in model.features.parameters():
            p.requires_grad = False
        warmup_opt = build_optimizer(model, params, stage="warmup")
        for ep in range(1, n_warmup + 1):
            train_loss, train_top1, train_top5 = run_epoch(
                model, train_loader, device,
                optimizer=warmup_opt, training=True,
                epoch_label=f"Warmup {ep}/{n_warmup}",
            )
            val_loss, val_top1, val_top5 = run_epoch(
                model, val_loader, device,
                training=False, epoch_label=f"Val {ep}",
            )
            mlflow.log_metrics(
                {
                    "train_loss": train_loss, "train_top1": train_top1, "train_top5": train_top5,
                    "val_loss": val_loss, "val_top1": val_top1, "val_top5": val_top5,
                },
                step=ep,
            )

        # ─── Stage 2: Finetune ───────────────────────────────────────────────
        for p in model.features.parameters():
            p.requires_grad = True
        finetune_opt = build_optimizer(model, params, stage="finetune")
        scheduler = build_scheduler(finetune_opt, params, n_finetune)
        for ep in range(1, n_finetune + 1):
            global_ep = n_warmup + ep
            train_loss, train_top1, train_top5 = run_epoch(
                model, train_loader, device,
                optimizer=finetune_opt, training=True,
                epoch_label=f"Finetune {ep}/{n_finetune}",
            )
            val_loss, val_top1, val_top5 = run_epoch(
                model, val_loader, device,
                training=False, epoch_label=f"Val {global_ep}",
            )
            scheduler.step()
            mlflow.log_metrics(
                {
                    "train_loss": train_loss, "train_top1": train_top1, "train_top5": train_top5,
                    "val_loss": val_loss, "val_top1": val_top1, "val_top5": val_top5,
                },
                step=global_ep,
            )

        torch.save(model.state_dict(), checkpoint_path)
        mlflow.log_artifact(str(checkpoint_path))
        logger.info("Final model saved to %s", checkpoint_path)

    return checkpoint_path


mlflow.set_tracking_uri(f"file://{MLRUNS_DIR}")
checkpoint_path = run_training(dataset=dataset, params=params)

## 11. Upload artifacts to S3 (checkpoint, class_names.json, mlruns/)

In [7]:
# # 1) Final model checkpoint
# model_uri = upload_file_to_s3(checkpoint_path)
# logger.info("Checkpoint uploaded to %s", model_uri)

# 2) Class names JSON
class_names_path = MODELS_DIR / "class_names.json"
save_idx_to_name(build_idx_to_name(dataset), class_names_path)
class_names_uri = upload_file_to_s3(class_names_path)
logger.info("Class names uploaded to %s", class_names_uri)

# # 3) Sync MLflow runs to S3 so they're viewable later
# upload_directory_to_s3(
#     local_dir=MLRUNS_DIR,
#     bucket=S3_BUCKET,
#     prefix=S3_MLRUNS_PREFIX,
#     region=AWS_REGION,
# )
# logger.info("MLflow runs synced to S3.")

NameError: name 'dataset' is not defined

## 12. Sync `mlruns/` from S3 back to local

Run this cell on your laptop (not inside SageMaker Studio) after training completes in the cloud. It pulls the MLflow run files down so you can view them locally with `mlflow ui --backend-store-uri ./mlruns`.

In [4]:
# Pull MLflow runs from S3 down to ./mlruns/ for local viewing.
# Uses the same download_directory_from_s3 helper defined earlier.
download_directory_from_s3(
    local_dir=MLRUNS_DIR,
    bucket=S3_BUCKET,
    prefix=S3_MLRUNS_PREFIX,
    region=AWS_REGION,
)
logger.info(
    "MLflow runs synced from s3://%s/%s to %s. "
    "View with:  mlflow ui --backend-store-uri %s",
    S3_BUCKET, S3_MLRUNS_PREFIX, MLRUNS_DIR, MLRUNS_DIR,
)

2026-05-13 18:16:58,783 INFO - Found credentials in shared credentials file: ~/.aws/credentials


2026-05-13 18:18:27,019 INFO - MLflow runs synced from s3://bird-ml-halajeel/mlruns to c:\Users\hamad\Desktop\Bird ML\notebooks\mlruns. View with:  mlflow ui --backend-store-uri c:\Users\hamad\Desktop\Bird ML\notebooks\mlruns
